In [0]:
# dbutils.fs.ls("abfss://source@mvnazureproject.dfs.core.windows.net/claims/")

In [0]:
# df_test = spark.read.option("header", "true").csv("abfss://source@mvnazureproject.dfs.core.windows.net/claims/claims_day2_incremental.csv")
# df_test.show(5)

In [0]:
# dbutils.fs.ls("abfss://bronze@mvnazureproject.dfs.core.windows.net/metadata/")

In [0]:
#from pyspark.sql.types import StructType, StructField, StringType

df_source = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("cloudFiles.schemaLocation", "abfss://bronze@mvnazureproject.dfs.core.windows.net/metadata/schemas/claims") \
    .load("abfss://source@mvnazureproject.dfs.core.windows.net/claims/") 

query_bronze = df_source.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://bronze@mvnazureproject.dfs.core.windows.net/metadata/checkpoints/claims") \
    .toTable("mvn_db_catalog.bronze.health_claims")

query_bronze.awaitTermination(timeout=30)
 #.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns") 

In [0]:
# display(spark.sql("""select * from mvn_db_catalog.bronze.health_claims"""))

In [0]:
from pyspark.sql.functions import col, get_json_object

# 1. Read from your raw Bronze table
df_bronze = spark.table("mvn_db_catalog.bronze.health_claims")

# 2. Extract 'Hospital' straight out of the JSON string inside _rescued_data
df_with_hospital = df_bronze.withColumn(
    "Hospital_Extracted", 
    get_json_object(col("_rescued_data"), "$.Hospital")
)

# 3. Apply cleaning: Deduplicate and cast types cleanly
df_silver_clean = df_with_hospital.dropDuplicates(["ClaimID"]) \
    .selectExpr(
        "cast(ClaimID as int) as ClaimID",
        "cast(MemberID as string) as MemberID",
        "cast(ProviderID as string) as ProviderID",
        "cast(Hospital_Extracted as string) as Hospital", # Saved cleanly to Silver!
        "cast(ClaimAmount as double) as ClaimAmount",
        "cast(ClaimDate as date) as ClaimDate",
        "cast(Status as string) as Status"
    )

# 4. Overwrite cleanly into your Silver layer Delta Table with schema merging enabled
df_silver_clean.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("mvn_db_catalog.silver.health_claims")

In [0]:
# display(spark.sql("""select * from mvn_db_catalog.silver.health_claims"""))

In [0]:
df_silver_clean.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("abfss://silver@mvnazureproject.dfs.core.windows.net/health_claims/")

In [0]:
# %sql DESCRIBE DETAIL mvn_db_catalog.gold.fact_claims;

In [0]:
# Read from clean Silver data
df_silver = spark.table("mvn_db_catalog.silver.health_claims")

# 1. Update dim_member (Overwrite is safe because it only keeps a unique list)
df_silver.select("MemberID").distinct() \
    .write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@mvnazureproject.dfs.core.windows.net/dim_member/") \
    .saveAsTable("mvn_db_catalog.gold.dim_member")

# 2. Update dim_provider (Overwrite is safe because it maps unique active entities)
df_silver.select("ProviderID", "Hospital").distinct() \
    .write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@mvnazureproject.dfs.core.windows.net/dim_provider/") \
    .saveAsTable("mvn_db_catalog.gold.dim_provider")

# 3. SMART UPSERT (MERGE) FOR FACT_CLAIMS
# Check if the fact table exists. If it doesn't, create it fresh.
if not spark.catalog.tableExists("mvn_db_catalog.gold.fact_claims"):
    df_silver.write.format("delta").mode("overwrite") \
        .option("path", "abfss://gold@mvnazureproject.dfs.core.windows.net/fact_claims/") \
        .saveAsTable("mvn_db_catalog.gold.fact_claims")
else:
    # If it already exists, run the MERGE to prevent duplicates
    df_silver.createOrReplaceTempView("incoming_silver_batch")
    spark.sql("""
        MERGE INTO mvn_db_catalog.gold.fact_claims AS target
        USING incoming_silver_batch AS source
        ON target.ClaimID = source.ClaimID
        WHEN MATCHED THEN
          UPDATE SET 
            target.MemberID = source.MemberID,
            target.ProviderID = source.ProviderID,
            target.Hospital = source.Hospital,
            target.ClaimAmount = source.ClaimAmount,
            target.ClaimDate = source.ClaimDate,
            target.Status = source.Status
        WHEN NOT MATCHED THEN
          INSERT *
    """)

# 4. Refresh your enhanced business summary (Overwrite mode is perfect here)
df_summary = spark.sql("""
    SELECT 
        Hospital,
        COUNT(ClaimID) as TotalClaims, 
        SUM(ClaimAmount) as TotalGrossAmount,
        SUM(CASE WHEN Status = 'Approved' THEN ClaimAmount ELSE 0 END) as ApprovedAmount,
        SUM(CASE WHEN Status = 'Rejected' THEN ClaimAmount ELSE 0 END) as RejectedAmount,
        SUM(CASE WHEN Status = 'Pending'  THEN ClaimAmount ELSE 0 END) as PendingAmount
    FROM mvn_db_catalog.gold.fact_claims 
    GROUP BY Hospital
""")

df_summary.write.format("delta").mode("overwrite") \
    .option("path", "abfss://gold@mvnazureproject.dfs.core.windows.net/hospital_summary_enhanced/") \
    .saveAsTable("mvn_db_catalog.gold.hospital_summary_enhanced")

In [0]:
display(spark.sql("""select * from mvn_db_catalog.gold.dim_member"""))
display(spark.sql("""select * from mvn_db_catalog.gold.dim_provider"""))
display(spark.sql("""select * from mvn_db_catalog.gold.fact_claims"""))
display(spark.sql("""select * from mvn_db_catalog.gold.hospital_summary_enhanced"""))
# #display(spark.sql("""select * from mvn_db_catalog.gold.provider_summary"""))